In [ ]:
# IMPORT LIBRARIES
import sys
import subprocess
import warnings
warnings.filterwarnings('ignore')

def ensure_package(pkg_name, import_name=None):
    """Install package if import fails."""
    import importlib
    try:
        importlib.import_module(import_name or pkg_name)
    except Exception:
        try:
            subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pkg_name])
        except Exception as e:
            print(f'Package install skipped/failed for {pkg_name}: {e}')

for pkg, imp in [
    ('pandas', 'pandas'),
    ('numpy', 'numpy'),
    ('matplotlib', 'matplotlib'),
    ('scikit-learn', 'sklearn'),
    ('seaborn', 'seaborn'),
    ('xgboost', 'xgboost'),
    ('shap', 'shap'),
]:
    ensure_package(pkg, imp)

import os
import re
import json
import math
import textwrap
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from IPython.display import display

try:
    from google.colab import files
    IN_COLAB = True
except Exception:
    IN_COLAB = False

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, average_precision_score,
    roc_curve, precision_recall_curve,
    confusion_matrix, classification_report
)
from sklearn.feature_selection import SelectKBest, mutual_info_classif
from sklearn.inspection import PartialDependenceDisplay, permutation_importance
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.base import clone
from xgboost import XGBClassifier
import shap

plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['axes.grid'] = True
sns.set_theme(style='whitegrid')
np.random.seed(42)
random.seed(42)

In [1]:
# UPLOADING CSV FILE
import pandas as pd
file_name = 'Foodpanda Analysis Dataset.csv'
print('\n' + '='*80)
print('STEP 2: UPLOAD CSV FILE')
print('='*80)
df = pd.read_csv(file_name)
print(f'Loaded file: {file_name}')
print(f'Dataset shape: {df.shape}')
display(df.head())


STEP 2: UPLOAD CSV FILE
Loaded file: Foodpanda Analysis Dataset.csv
Dataset shape: (6000, 21)


,customer_id,gender,age,city,signup_date,order_id,order_date,restaurant_name,dish_name,category,...,price,payment_method,order_frequency,last_order_date,loyalty_points,churned,rating,rating_date,delivery_status,Unnamed: 20
0,C5221,Male,Senior,Lahore,10/3/2023,O9221,1/10/2024,McDonald's,Burger,Italian,...,1291.14,Card,7,8/21/2025,42,Inactive,3,11/29/2024,Cancelled,NaN
1,C2831,Male,Adult,Multan,7/7/2024,O6831,8/23/2023,KFC,Burger,Italian,...,956.04,Wallet,24,11/25/2024,81,Active,2,8/21/2025,Delayed,NaN
2,C2851,Other,Senior,Multan,6/20/2025,O6851,8/23/2023,Pizza Hut,Fries,Italian,...,882.51,Cash,42,5/10/2025,82,Inactive,3,9/19/2024,Delayed,NaN
3,C1694,Female,Senior,Peshawar,9/5/2023,O5694,8/23/2023,Subway,Pizza,Dessert,...,231.30,Card,27,7/24/2025,45,Inactive,2,6/29/2025,Delayed,NaN
4,C5641,Other,Senior,Islamabad,10/13/2023,O9641,3/27/2024,Pizza Hut,Burger,Chinese,...,620.96,Card,17,8/21/2025,206,Active,2,4/9/2025,Cancelled,NaN


In [2]:
# Summarize feature types, missing values, outliers, etc.

import sys
import subprocess
import warnings
warnings.filterwarnings('ignore')

def ensure_package(pkg_name, import_name=None):
    """Install package if import fails."""
    import importlib
    try:
        importlib.import_module(import_name or pkg_name)
    except Exception:
        try:
            subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pkg_name])
        except Exception as e:
            print(f'Package install skipped/failed for {pkg_name}: {e}')

for pkg, imp in [
    ('pandas', 'pandas'),
    ('numpy', 'numpy'),
    ('matplotlib', 'matplotlib'),
    ('scikit-learn', 'sklearn'),
    ('seaborn', 'seaborn'),
    ('xgboost', 'xgboost'),
    ('shap', 'shap'),
]:
    ensure_package(pkg, imp)

import os
import re
import json
import math
import textwrap
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from IPython.display import display

try:
    from google.colab import files
    IN_COLAB = True
except Exception:
    IN_COLAB = False

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, average_precision_score,
    roc_curve, precision_recall_curve,
    confusion_matrix, classification_report
)
from sklearn.feature_selection import SelectKBest, mutual_info_classif
from sklearn.inspection import PartialDependenceDisplay, permutation_importance
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.base import clone
from xgboost import XGBClassifier
import shap

plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['axes.grid'] = True
sns.set_theme(style='whitegrid')
np.random.seed(42)
random.seed(42)

def detect_outliers_iqr(series):
    q1 = series.quantile(0.25)
    q3 = series.quantile(0.75)
    iqr = q3 - q1
    lower_bound = q1 - 1.5 * iqr
    upper_bound = q3 + 1.5 * iqr
    outliers = series[(series < lower_bound) | (series > upper_bound)]
    return {
        'q1': q1,
        'q3': q3,
        'iqr': iqr,
        'lower_bound': lower_bound,
        'upper_bound': upper_bound,
        'outlier_count': len(outliers),
        'outlier_pct': (len(outliers) / len(series)) * 100 if len(series) > 0 else 0
    }

def safe_to_datetime(series):
    return pd.to_datetime(series, errors='coerce')

print('STEP 2: DATA QUALITY SUMMARY')

feature_summary = pd.DataFrame({
    'variable': df.columns,
    'dtype': df.dtypes.astype(str).values,
    'missing_count': df.isna().sum().values,
    'missing_pct': (df.isna().mean().values * 100).round(2),
    'n_unique': [df[c].nunique(dropna=True) for c in df.columns]
})
display(feature_summary)

numeric_cols_raw = df.select_dtypes(include=[np.number]).columns.tolist()
outlier_rows = []
for col in numeric_cols_raw:
    outlier_info = detect_outliers_iqr(df[col].dropna())
    outlier_info['variable'] = col
    outlier_rows.append(outlier_info)
outlier_df = pd.DataFrame(outlier_rows)[[
    'variable', 'q1', 'q3', 'iqr', 'lower_bound', 'upper_bound', 'outlier_count', 'outlier_pct'
]]
print('\nOutlier summary (IQR method):')
display(outlier_df)

# Date integrity checks
potential_date_cols = [c for c in df.columns if 'date' in c.lower()]
date_issue_log = []
for c in potential_date_cols:
    parsed = safe_to_datetime(df[c])
    date_issue_log.append({
        'date_column': c,
        'unparseable_dates': int(parsed.isna().sum()),
        'min_date': parsed.min(),
        'max_date': parsed.max()
    })
date_issue_df = pd.DataFrame(date_issue_log)
print('\nDate parsing summary:')
display(date_issue_df)

# Detect potentially problematic columns and data leakage
problematic_notes = []
all_nan_cols = [c for c in df.columns if df[c].isna().all()]
if all_nan_cols:
    problematic_notes.append(f"Columns with all missing values: {all_nan_cols} -> drop them.")

high_cardinality_cols = [c for c in df.select_dtypes(include='object').columns if df[c].nunique() > max(100, 0.2 * len(df))]
if high_cardinality_cols:
    problematic_notes.append(f"High-cardinality identifiers/text columns: {high_cardinality_cols} -> usually drop from modeling.")

# Specific chronology issues if the expected columns exist
for a, b, note in [
    ('signup_date', 'order_date', 'signup_date occurs after order_date'),
    ('last_order_date', 'order_date', 'last_order_date occurs after order_date'),
    ('rating_date', 'order_date', 'rating_date occurs before order_date')
]:
    if a in df.columns and b in df.columns:
        a_dt = safe_to_datetime(df[a])
        b_dt = safe_to_datetime(df[b])
        if note == 'signup_date occurs after order_date':
            count = int((a_dt > b_dt).sum())
        elif note == 'last_order_date occurs after order_date':
            count = int((a_dt > b_dt).sum())
        else:
            count = int((a_dt < b_dt).sum())
        problematic_notes.append(f"Chronology check: {note} in {count} rows.")

# Leakage notes
for leakage_col in ['rating', 'rating_date', 'churned']:
    if leakage_col in df.columns:
        problematic_notes.append(
            f"Potential target leakage: '{leakage_col}' may only be known after delivery/cancellation outcome -> drop for pre-fulfillment prediction."
        )

print('\nProblematic data findings and recommendations:')
for idx, note in enumerate(problematic_notes, start=1):
    print(f'{idx}. {note}')

print('\nRecommended cleaning actions implemented later in this script:')
print('1. Drop empty or useless columns such as unnamed/all-missing columns.')
print('2. Parse dates safely and create timing-based features known before fulfillment.')
print('3. Remove identifier columns from modeling (for example customer_id and order_id).')
print('4. Drop likely leakage columns only known after fulfillment (for example rating, rating_date, churned).')
print('5. Cap numeric outliers using the IQR method instead of deleting many rows.')

STEP 2: DATA QUALITY SUMMARY


,variable,dtype,missing_count,missing_pct,n_unique
0,customer_id,object,0,0.00,6000
1,gender,object,0,0.00,3
2,age,object,0,0.00,3
3,city,object,0,0.00,5
4,signup_date,object,0,0.00,730
5,order_id,object,0,0.00,6000
6,order_date,object,0,0.00,730
7,restaurant_name,object,0,0.00,5
8,dish_name,object,0,0.00,5
9,category,object,0,0.00,5



Outlier summary (IQR method):


,variable,q1,q3,iqr,lower_bound,upper_bound,outlier_count,outlier_pct
0,quantity,2.0000,4.0000,2.00,-1.0000,7.0000,0,0.0
1,price,441.9975,1149.7375,707.74,-619.6125,2211.3475,0,0.0
2,order_frequency,13.0000,37.0000,24.00,-23.0000,73.0000,0,0.0
3,loyalty_points,125.0000,378.0000,253.00,-254.5000,757.5000,0,0.0
4,rating,2.0000,4.0000,2.00,-1.0000,7.0000,0,0.0
5,Unnamed: 20,0.3280,0.3280,0.00,0.3280,0.3280,0,0.0



Date parsing summary:


,date_column,unparseable_dates,min_date,max_date
0,signup_date,0,2023-08-22,2025-08-21
1,order_date,0,2023-08-23,2025-08-22
2,last_order_date,0,2024-08-21,2025-08-21
3,rating_date,0,2024-08-21,2025-08-21



Problematic data findings and recommendations:
1. High-cardinality identifiers/text columns: ['customer_id', 'order_id'] -> usually drop from modeling.
2. Chronology check: signup_date occurs after order_date in 3028 rows.
3. Chronology check: last_order_date occurs after order_date in 4514 rows.
4. Chronology check: rating_date occurs before order_date in 1507 rows.
5. Potential target leakage: 'rating' may only be known after delivery/cancellation outcome -> drop for pre-fulfillment prediction.
6. Potential target leakage: 'rating_date' may only be known after delivery/cancellation outcome -> drop for pre-fulfillment prediction.
7. Potential target leakage: 'churned' may only be known after delivery/cancellation outcome -> drop for pre-fulfillment prediction.

Recommended cleaning actions implemented later in this script:
1. Drop empty or useless columns such as unnamed/all-missing columns.
2. Parse dates safely and create timing-based features known before fulfillment.
3. Remove id